# Alpha draft model experiment

Load normalized Dota 2 drafts, build PyTorch tensors, create the Alpha model, and prepare a train/validation loop.

In [28]:
import importlib
import json
import sys
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

import scripts.ml.dataset as dataset_module
import scripts.ml.models.alpha as alpha_module

importlib.reload(dataset_module)
importlib.reload(alpha_module)

DatasetMaker = dataset_module.DatasetMaker
AlphaDraftModel = alpha_module.AlphaDraftModel

In [29]:
DATASET_SIZE = 1_000
BATCH_SIZE = 64
VAL_FRACTION = 0.2
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [30]:
with DatasetMaker(env_file=str(PROJECT_ROOT / ".env")) as dataset:
    matches = dataset.fetch_normalized_match_drafts(DATASET_SIZE)

len(matches), matches[0]

(1000,
 NormalizedMatchDraft(match_id=8797782323, radiant_hero_ids=[138, 23, 69, 22, 40], dire_hero_ids=[38, 78, 73, 123, 86], winner_side=1))

In [31]:
radiant_ids = torch.tensor([match.radiant_hero_ids for match in matches], dtype=torch.long)
dire_ids = torch.tensor([match.dire_hero_ids for match in matches], dtype=torch.long)

# DatasetMaker encodes winner_side as radiant=0, dire=1.
# With these labels, sigmoid(logit) is the model's probability of Dire win.
labels = torch.tensor([match.winner_side for match in matches], dtype=torch.float32)

radiant_ids.shape, dire_ids.shape, labels.shape, labels.float().mean()

(torch.Size([1000, 5]),
 torch.Size([1000, 5]),
 torch.Size([1000]),
 tensor(0.4760))

In [33]:
dataset_tensor = TensorDataset(radiant_ids, dire_ids, labels)

val_size = max(1, int(len(dataset_tensor) * VAL_FRACTION))
train_size = len(dataset_tensor) - val_size

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(dataset_tensor, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

len(train_dataset), len(val_dataset)

(800, 200)

In [34]:
heroes_path = PROJECT_ROOT / "dotaconstants" / "build" / "heroes.json"
heroes = json.loads(heroes_path.read_text(encoding="utf-8"))
num_heroes = max(int(hero["id"]) for hero in heroes.values()) + 1

model = AlphaDraftModel(num_heroes=num_heroes, embedding_dim=32).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_heroes, model

(156,
 AlphaDraftModel(
   (hero_embedding): Embedding(156, 32)
   (classifier): Sequential(
     (0): Linear(in_features=192, out_features=128, bias=True)
     (1): ReLU()
     (2): Dropout(p=0.2, inplace=False)
     (3): Linear(in_features=128, out_features=64, bias=True)
     (4): ReLU()
     (5): Dropout(p=0.1, inplace=False)
     (6): Linear(in_features=64, out_features=1, bias=True)
   )
 ))

In [35]:
batch_radiant_ids, batch_dire_ids, batch_labels = next(iter(train_loader))
batch_radiant_ids = batch_radiant_ids.to(device)
batch_dire_ids = batch_dire_ids.to(device)
batch_labels = batch_labels.to(device)

model.train()
logits = model(batch_radiant_ids, batch_dire_ids)
loss = criterion(logits, batch_labels)
probs = torch.sigmoid(logits)

logits.shape, loss.item(), probs[:5].detach().cpu()

(torch.Size([64]),
 0.6793106198310852,
 tensor([0.3629, 0.5103, 0.4323, 0.4599, 0.3696]))

In [36]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for radiant_batch, dire_batch, label_batch in loader:
        radiant_batch = radiant_batch.to(device)
        dire_batch = dire_batch.to(device)
        label_batch = label_batch.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(radiant_batch, dire_batch)
        loss = criterion(logits, label_batch)
        loss.backward()
        optimizer.step()

        batch_size = label_batch.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for radiant_batch, dire_batch, label_batch in loader:
        radiant_batch = radiant_batch.to(device)
        dire_batch = dire_batch.to(device)
        label_batch = label_batch.to(device)

        logits = model(radiant_batch, dire_batch)
        loss = criterion(logits, label_batch)
        preds = (torch.sigmoid(logits) >= 0.5).float()

        batch_size = label_batch.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (preds == label_batch).sum().item()
        total_examples += batch_size

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }

In [37]:
EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device)
    print(
        f"epoch={epoch:02d} "
        f"train_loss={train_loss:.4f} "
        f"val_loss={val_metrics['loss']:.4f} "
        f"val_accuracy={val_metrics['accuracy']:.3f}"
    )

epoch=01 train_loss=0.7115 val_loss=0.6853 val_accuracy=0.535
epoch=02 train_loss=0.6466 val_loss=0.6846 val_accuracy=0.550
epoch=03 train_loss=0.5981 val_loss=0.6868 val_accuracy=0.535
epoch=04 train_loss=0.5547 val_loss=0.6955 val_accuracy=0.575
epoch=05 train_loss=0.4944 val_loss=0.7211 val_accuracy=0.590
epoch=06 train_loss=0.4271 val_loss=0.7496 val_accuracy=0.565
epoch=07 train_loss=0.3622 val_loss=0.8041 val_accuracy=0.590
epoch=08 train_loss=0.3003 val_loss=0.8389 val_accuracy=0.585
epoch=09 train_loss=0.2212 val_loss=0.8916 val_accuracy=0.580
epoch=10 train_loss=0.1818 val_loss=0.9719 val_accuracy=0.595


In [39]:
@torch.no_grad()
def confusion_matrix(model, loader, device):
    model.eval()

    tp = tn = fp = fn = 0

    for radiant_batch, dire_batch, label_batch in loader:
        radiant_batch = radiant_batch.to(device)
        dire_batch = dire_batch.to(device)
        label_batch = label_batch.to(device).long()

        logits = model(radiant_batch, dire_batch)
        preds = (torch.sigmoid(logits) >= 0.5).long()

        tp += ((preds == 1) & (label_batch == 1)).sum().item()
        tn += ((preds == 0) & (label_batch == 0)).sum().item()
        fp += ((preds == 1) & (label_batch == 0)).sum().item()
        fn += ((preds == 0) & (label_batch == 1)).sum().item()

    return {
        "true_radiant_pred_radiant": tn,
        "true_radiant_pred_dire": fp,
        "true_dire_pred_radiant": fn,
        "true_dire_pred_dire": tp,
    }

confusion_matrix(model, val_loader, device)

{'true_radiant_pred_radiant': 75,
 'true_radiant_pred_dire': 31,
 'true_dire_pred_radiant': 50,
 'true_dire_pred_dire': 44}

## Manual draft prediction

In [40]:
def predict_manual_draft(model, radiant_heroes, dire_heroes, *, env_file=str(PROJECT_ROOT / ".env")):
    if len(radiant_heroes) != 5:
        raise ValueError(f"radiant_heroes must contain 5 heroes, got {len(radiant_heroes)}")
    if len(dire_heroes) != 5:
        raise ValueError(f"dire_heroes must contain 5 heroes, got {len(dire_heroes)}")

    with DatasetMaker(env_file=env_file) as dataset:
        radiant_ids = [dataset.hero_name_to_id(hero) for hero in radiant_heroes]
        dire_ids = [dataset.hero_name_to_id(hero) for hero in dire_heroes]

    radiant_tensor = torch.tensor([radiant_ids], dtype=torch.long, device=device)
    dire_tensor = torch.tensor([dire_ids], dtype=torch.long, device=device)

    model.eval()
    with torch.no_grad():
        logit = model(radiant_tensor, dire_tensor).item()
        p_dire = torch.sigmoid(torch.tensor(logit)).item()

    return {
        "radiant_ids": radiant_ids,
        "dire_ids": dire_ids,
        "logit": logit,
        "p_radiant": 1.0 - p_dire,
        "p_dire": p_dire,
        "predicted_winner": "dire" if p_dire >= 0.5 else "radiant",
    }

In [65]:
radiant_heroes = [
    "keeper of the light",
    "axe",
    "phantom assassin",
    "ogre magi",
    "lion",
]

dire_heroes = [
    "bristleback",
    "huskar",
    "phoenix",
    "ember spirit",
    "juggernaut",
]

predict_manual_draft(model, radiant_heroes, dire_heroes)

{'radiant_ids': [90, 2, 44, 84, 26],
 'dire_ids': [99, 59, 110, 106, 8],
 'logit': 0.20899930596351624,
 'p_radiant': 0.44793957471847534,
 'p_dire': 0.5520604252815247,
 'predicted_winner': 'dire'}